In [12]:
"""
week3_task2.py
=================
Selective refresh of the Week 2 analytical data products after
week3_incremental.py has ingested new raw data / evolved schemas.


1. Dependency map
Each data product is mapped to what delta tables it is built on, so when they change they are refreshed. This prevents unnecessary recomputation if 
an unrelated table gets changed.

2. Change detection
   A tiny JSON state file (pipeline_state.json) stores, per monitored raw
   Delta table, the Delta-log version and column list we last saw.
   A table counts as "changed" if the version bumped (new rows) OR the
   column list differs (schema evolution) -- both need a downstream refresh.
   On the very first run there is no baseline yet, so we just record the
   fingerprint and do nothing else (Week 1/2 already built everything once).

3. Integrated tables refresh
   - New taxi rows are enriched exactly like the Week 1 integration logic
     and APPENDED to the two integrated Delta tables (by_date / by_borough).
   - If weather or air_quality themselves changed, we do NOT append -- that
     would duplicate fact rows. Instead we run a targeted Delta MERGE that
     backfills weather_*/air_q_* columns only on the existing integrated
     rows that are still NULL there (i.e. trips whose hour didn't have a
     weather/air-quality reading yet).

4. Data product refreshes
   - taxi_zone_statistics / daily_mobility_summary: refreshed INCREMENTALLY.
     We aggregate only the new taxi rows and merge (sum) that tiny result
     into the existing (also tiny) summary table, then recompute the
     avg columns. We never rescan the full taxi history.
   - weather_impact_summary / air_quality_impact_summary: on a dependency
     change we recompute that ONE product from the (now backfilled)
     integrated table. These are cheap 2-row / few-hundred-row aggregates
     (seconds, per the Week 2 benchmarks), so a full recompute of just
     this one product is simpler than incremental backfill bookkeeping
     and still satisfies "only refresh what's affected".

"""

import json
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable


spark = (
    SparkSession.builder
    .appName("SelectiveRefresh")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

STATE_PATH = "pipeline_state.json"

# Raw tables we watch for changes. taxi_trips_03 is the only taxi table
# week3_incremental.py ever appends new rows to, so that's the one we track.
RAW_TABLES = {
    "weather": "delta/weather",
    "air_quality": "delta/air_quality",
    "taxi_trips_03": "delta/taxi_trips_03",
}

PRODUCT_DEPENDENCIES = {
    "taxi_zone_statistics":       {"taxi_trips_03", "taxi_zone_lookup"},
    "daily_mobility_summary":     {"taxi_trips_03"},
    "weather_impact_summary":     {"taxi_trips_03", "weather"},
    "air_quality_impact_summary": {"taxi_trips_03", "air_quality"},
}

INTEGRATED_TABLES = [
    "delta/integrated_taxi_trips_by_date",
    "delta/integrated_taxi_trips_by_borough",
]

PRODUCTS_PATH = "delta/assignment4_task4"

# Helper: Fetch existing metadata from a data product
def get_table_metadata(path):
    """
    Reads the existing Delta table to retrieve the original creation_time 
    and the current schema_version. If the table doesn't exist yet, 
    returns None and '0.0' so the first run defaults to current_time and '1.0'.
    """
    try:
        df = spark.read.format("delta").load(path)
        # Grab exactly one row to read the metadata
        row = df.select("creation_time", "schema_version").limit(1).collect()
        if row:
            old_creation = row[0]["creation_time"]
            old_version = float(row[0]["schema_version"])
            return old_creation, old_version
    except Exception:
        # Table doesn't exist yet (first time running)
        pass
    
    return None, 0.0
    

#change detection

def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, "r") as f:
            return json.load(f)
    return {"tables": {}}


def save_state(state):
    with open(STATE_PATH, "w") as f:
        json.dump(state, f, indent=2)


def table_fingerprint(path):
    dt = DeltaTable.forPath(spark, path)
    version = dt.history(1).select("version").collect()[0]["version"]
    columns = spark.read.format("delta").load(path).columns
    return version, columns


def detect_changes(state):
    """Returns (changes, is_bootstrap).
    changes: {table_name: {"old_version", "new_version", "new_columns"}}
    """
    is_bootstrap = len(state["tables"]) == 0
    changes = {}
    for name, path in RAW_TABLES.items():
        version, columns = table_fingerprint(path)
        prev = state["tables"].get(name)
        if prev is None:
            # First time we've ever seen this table -- nothing to diff
            # against yet, baseline gets written at the end of main().
            continue
        new_cols = [c for c in columns if c not in prev["columns"]]
        if version != prev["version"] or new_cols:
            changes[name] = {
                "old_version": prev["version"],
                "new_version": version,
                "new_columns": new_cols,
            }
    return changes, is_bootstrap


def get_new_rows(path, old_version):
    """Rows present now but not in the snapshot at old_version.
    Works because every table we track here is insert-only (appends /
    whenNotMatchedInsertAll), so a version-diff anti-join == 'the new rows'.
    """
    current = spark.read.format("delta").load(path)
    old_snapshot = spark.read.format("delta").option("versionAsOf", old_version).load(path)
    common_cols = old_snapshot.columns
    return current.join(current.select(common_cols).exceptAll(old_snapshot.select(common_cols)).select(common_cols),
                         on=common_cols, how="inner").dropDuplicates()


def affected_products(changed_table_names):
    changed = set(changed_table_names)
    return [p for p, deps in PRODUCT_DEPENDENCIES.items() if deps & changed]


def is_empty(df):
    return len(df.take(1)) == 0



# Integration helpers (mirrors week1's integration logic, but scoped to
# just the rows/keys that actually need touching)

def rename_all(df, prefix):
    for c in df.columns:
        df = df.withColumnRenamed(c, prefix + c)
    return df

#build the new integrated table
def build_integration_increment(new_taxi_rows):
    """Enrich only the new taxi rows with the CURRENT weather / zone /
    air-quality tables, exactly like week1's full-table integration."""
    weather = rename_all(spark.read.format("delta").load("delta/weather"), "weather_")

    lookup = spark.read.format("delta").load("delta/taxi_zone_lookup").drop("service_zone")
    lookup_pu = (lookup
                 .withColumnRenamed("borough", "pu_borough")
                 .withColumnRenamed("zone", "pu_zone"))
    lookup_do = (lookup
                 .withColumnRenamed("borough", "do_borough")
                 .withColumnRenamed("zone", "do_zone"))

    air_quality_ny = rename_all(
        spark.read.format("delta").load("delta/air_quality").filter(F.col("state_code") == 36),
        "air_q_",
    )

    df = (
        new_taxi_rows
        .withColumn("weather_time", F.date_trunc("hour", F.col("tpep_pickup_datetime")))
        .join(weather, F.col("weather_time") == F.col("weather_timestamp"), "left")
        .drop("weather_timestamp")
        .join(lookup_pu, F.col("pu_location_id") == F.col("location_id"), "left")
        .drop("location_id")
        .join(lookup_do, F.col("do_location_id") == F.col("location_id"), "left")
        .drop("location_id")
        .join(
            air_quality_ny,
            (F.col("pu_borough") == F.col("air_q_county_name"))
            & (F.col("weather_time") == F.col("air_q_datetime_local")),
            "left",
        )
        .drop("air_q_county_name", "air_q_datetime_local")
        .withColumn("day_timestamp", F.date_trunc("day", F.col("weather_time")))
    )
    return df

#combine the old and new integration tables
def append_integration_increment(increment_df):
    from pyspark.sql import functions as F
    
    # 1. Read the schema of the target table
    target_path = "delta/integrated_taxi_trips_by_date"
    target_schema = spark.read.format("delta").load(target_path).schema
    
    # 2. Force the incoming dataframe to match the target table's data types
    for field in target_schema.fields:
        if field.name in increment_df.columns:
            increment_df = increment_df.withColumn(
                field.name, 
                F.col(field.name).cast(field.dataType)
            )

    print(f" Appending {increment_df.count():,} new rows to the integrated tables ...")
    
    # 3. Save as normal
    (increment_df.write.format("delta").mode("append")
     .option("mergeSchema", "true")
     .partitionBy("day_timestamp")
     .save(target_path))
     
    (increment_df.write.format("delta").mode("append")
     .option("mergeSchema", "true")
     .partitionBy("pu_borough")
     .save("delta/integrated_taxi_trips_by_borough"))



def backfill_weather(new_weather_rows):
    """Fill weather_* columns on existing integrated rows that were
    previously unmatched (weather_temp IS NULL) and now have a match."""
    if is_empty(new_weather_rows):
        return
    new_weather = rename_all(new_weather_rows, "weather_")
    set_cols = [c for c in new_weather.columns if c != "weather_timestamp"]
    print(f"  Backfilling weather onto previously-unmatched integrated rows ...")
    for path in INTEGRATED_TABLES:
        target = DeltaTable.forPath(spark, path)
        (target.alias("t")
            .merge(
                new_weather.alias("w"),
                "t.weather_time = w.weather_timestamp AND t.weather_temp IS NULL",
            )
            .whenMatchedUpdate(set={c: f"w.{c}" for c in set_cols})
            .execute())


def backfill_air_quality(new_air_quality_rows):
    """Fill air_q_* columns on existing integrated rows that were
    previously unmatched (air_q_sample_measurement IS NULL)."""
    if is_empty(new_air_quality_rows):
        return
    ny = rename_all(new_air_quality_rows.filter(F.col("state_code") == 36), "air_q_")
    if is_empty(ny):
        return
    set_cols = [c for c in ny.columns if c not in ("air_q_county_name", "air_q_datetime_local")]
    print(f"  Backfilling air quality onto previously-unmatched integrated rows ...")
    for path in INTEGRATED_TABLES:
        target = DeltaTable.forPath(spark, path)
        (target.alias("t")
            .merge(
                ny.alias("a"),
                "t.pu_borough = a.air_q_county_name "
                "AND t.weather_time = a.air_q_datetime_local "
                "AND t.air_q_sample_measurement IS NULL",
            )
            .whenMatchedUpdate(set={c: f"a.{c}" for c in set_cols})
            .execute())


# ──────────────────────────────────────────────────────────────────────────
# Product refresh: incremental merge-sum (taxi-only products)
# ──────────────────────────────────────────────────────────────────────────

#Merging the old and new values for incrememnting     
def merge_sum_increment(product_name, increment_df, keys, sum_cols, avg_pairs):
    path = f"{PRODUCTS_PATH}/{product_name}"
    
    # 1. Fetch existing metadata
    old_creation, old_version = get_table_metadata(path)
    
    # 2. Determine new metadata values
    new_version_str = f"{old_version + 1.0:.1f}"  # Increments 1.0 -> 2.0. (Use + 0.1 for 1.1)
    creation_col = F.lit(old_creation).cast("timestamp") if old_creation else F.current_timestamp()

    # 3. Read existing data and process
    existing = spark.read.format("delta").load(path).select(*keys, *sum_cols)
    combined = existing.unionByName(increment_df.select(*keys, *sum_cols))
    merged = combined.groupBy(*keys).agg(*[F.sum(c).alias(c) for c in sum_cols])

    for avg_col, num_col, denom_col in avg_pairs:
        merged = merged.withColumn(avg_col, F.round(F.col(num_col) / F.col(denom_col)))

    # 4. Apply the preserved/incremented metadata
    merged = (merged
        .withColumn("data_source", F.lit(f"incremental_merge:{path}"))
        .withColumn("creation_time", creation_col)
        .withColumn("refresh_time", F.current_timestamp())
        .withColumn("schema_version", F.lit(new_version_str)))

    merged.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)



# incremental_daily_mobility and incremental_taxi_zone_stats remain unchanged 
# since they just call merge_sum_increment


def incremental_daily_mobility(new_taxi_rows):
    inc = (
        new_taxi_rows
        .select("tpep_pickup_datetime", "trip_distance")
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
        .groupBy("pickup_hour", "day_of_week")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.sum("trip_distance").alias("total_trip_distance"),
        )
    )
    merge_sum_increment(
        "daily_mobility_summary", inc,
        keys=["pickup_hour", "day_of_week"],
        sum_cols=["taxi_demand", "total_trip_distance"],
        avg_pairs=[("avg_trip_distance", "total_trip_distance", "taxi_demand")],
    )


def incremental_taxi_zone_stats(new_taxi_rows, zone_lookup):
    lookup_pu = (zone_lookup
                 .select("location_id", "borough", "zone")
                 .withColumnRenamed("borough", "pu_borough")
                 .withColumnRenamed("zone", "pu_zone"))

    inc = (
        new_taxi_rows
        .join(lookup_pu, new_taxi_rows.pu_location_id == lookup_pu.location_id, "left")
        .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
        .groupBy("pu_zone", "month")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.sum("trip_distance").alias("total_trip_distance"),
            F.sum("fare_amount").alias("total_fare_amount"),
        )
    )
    merge_sum_increment(
        "taxi_zone_statistics", inc,
        keys=["pu_zone", "month"],
        sum_cols=["taxi_demand", "total_trip_distance", "total_fare_amount"],
        avg_pairs=[
            ("avg_trip_distance", "total_trip_distance", "taxi_demand"),
            ("avg_fare_amount", "total_fare_amount", "taxi_demand"),
        ],
    )



# Product refresh: full recompute

def refresh_weather_impact_summary():
    path = f"{PRODUCTS_PATH}/weather_impact_summary"
    
    # 1. Fetch existing metadata
    old_creation, old_version = get_table_metadata(path)
    new_version_str = f"{old_version + 1.0:.1f}"
    creation_col = F.lit(old_creation).cast("timestamp") if old_creation else F.current_timestamp()

    integrated = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")
    df = (
        integrated
        .select("trip_distance", "weather_prcp")
        .dropna()
        .withColumn("is_raining", F.col("weather_prcp") != 0.0)
        .groupBy("is_raining")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
            F.round(F.sum("trip_distance")).alias("total_trip_distance"),
        )
        # 2. Apply metadata
        .withColumn("data_source", F.lit("delta/integrated_taxi_trips_by_borough"))
        .withColumn("creation_time", creation_col)
        .withColumn("refresh_time", F.current_timestamp())
        .withColumn("schema_version", F.lit(new_version_str))
    )
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)

    
def refresh_air_quality_impact_summary():
    path = f"{PRODUCTS_PATH}/air_quality_impact_summary"
    
    # 1. Fetch existing metadata
    old_creation, old_version = get_table_metadata(path)
    new_version_str = f"{old_version + 1.0:.1f}"
    creation_col = F.lit(old_creation).cast("timestamp") if old_creation else F.current_timestamp()

    integrated = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")
    df = (
        integrated
        .select("tpep_pickup_datetime", "air_q_sample_measurement", "trip_distance")
        .dropna(subset=["tpep_pickup_datetime", "air_q_sample_measurement"])
        .withColumn("pickup_hour", F.date_trunc("hour", "tpep_pickup_datetime"))
        .groupBy("pickup_hour")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.avg("air_q_sample_measurement").alias("avg_air_quality"),
            F.round(F.avg("trip_distance")).alias("avg_trip_distance"),
            F.round(F.sum("trip_distance")).alias("total_trip_distance"),
        )
        # 2. Apply metadata
        .withColumn("data_source", F.lit("delta/integrated_taxi_trips_by_borough"))
        .withColumn("creation_time", creation_col)
        .withColumn("refresh_time", F.current_timestamp())
        .withColumn("schema_version", F.lit(new_version_str))
    )
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)




def main():
    t0 = time.time()
    state = load_state()
    changes, is_bootstrap = detect_changes(state)

    if is_bootstrap:
        print("No prior state found -- recording baseline fingerprints only "
              "(Week 1/2 already built everything from scratch once).")
        for name, path in RAW_TABLES.items():
            version, columns = table_fingerprint(path)
            state["tables"][name] = {"version": version, "columns": columns}
        save_state(state)
        return

    if not changes:
        print("No changes detected in any monitored raw table -- nothing to refresh.")
        return

    print("=" * 60)
    print("Changed raw tables:", list(changes.keys()))
    for name, info in changes.items():
        if info["new_columns"]:
            print(f"  {name}: schema evolved, new columns = {info['new_columns']}")
    print("=" * 60)

    new_taxi = None

    # ── Taxi trips: append new fact rows into the integrated tables ──────
    if "taxi_trips_03" in changes:
        new_taxi = get_new_rows("delta/taxi_trips_03", changes["taxi_trips_03"]["old_version"]).cache()
        n = new_taxi.count()
        print(f"New taxi trip rows: {n:,}")
        if n > 0:
            increment = build_integration_increment(new_taxi)
            append_integration_increment(increment)

    # ── Weather / air quality: backfill previously-unmatched rows ────────
    if "weather" in changes:
        new_weather = get_new_rows("delta/weather", changes["weather"]["old_version"])
        backfill_weather(new_weather)

    if "air_quality" in changes:
        new_air_quality = get_new_rows("delta/air_quality", changes["air_quality"]["old_version"])
        backfill_air_quality(new_air_quality)

    # ── Refresh only the products whose dependencies actually changed ────
    to_refresh = affected_products(changes.keys())
    print("Products to refresh:", to_refresh if to_refresh else "(none)")

    zone_lookup = spark.read.format("delta").load("delta/taxi_zone_lookup")

    for product in to_refresh:
        print(f"  Refreshing {product} ...")
        if product == "daily_mobility_summary" and new_taxi is not None:
            incremental_daily_mobility(new_taxi)
        elif product == "taxi_zone_statistics" and new_taxi is not None:
            incremental_taxi_zone_stats(new_taxi, zone_lookup)
        elif product == "weather_impact_summary":
            refresh_weather_impact_summary()
        elif product == "air_quality_impact_summary":
            refresh_air_quality_impact_summary()

    if new_taxi is not None:
        new_taxi.unpersist()

    # ── Update state with new fingerprints for everything we track ───────
    for name, path in RAW_TABLES.items():
        version, columns = table_fingerprint(path)
        state["tables"][name] = {"version": version, "columns": columns}
    save_state(state)

    print(f"Done in {round(time.time() - t0, 2)} s.")


if __name__ == "__main__":
    main()

Changed raw tables: ['weather', 'air_quality', 'taxi_trips_03']
  weather: schema evolved, new columns = ['humidity']
  air_quality: schema evolved, new columns = ['aqi']


New taxi trip rows: 220,984
 Appending 242,706 new rows to the integrated tables ...


Products to refresh: ['taxi_zone_statistics', 'daily_mobility_summary', 'weather_impact_summary', 'air_quality_impact_summary']
  Refreshing taxi_zone_statistics ...


  Refreshing daily_mobility_summary ...


  Refreshing weather_impact_summary ...
  Refreshing air_quality_impact_summary ...
Done in 51.69 s.
